In [1]:
!pip install -Uqq sae-lens transformers accelerate pandas scipy "numpy<2.0.0" "bitsandbytes>=0.46.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:
    hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found. Set it in Kaggle Secrets or a local .env file.")

login(token=hf_token)

In [ ]:
import os

IS_KAGGLE = os.path.exists("/kaggle/input")

BASE_DATA_PATH = "/kaggle/input/datasets/shaurya01pratap/principledata/New Clean Data/" if IS_KAGGLE else "../data/principledata/New Clean Data/"
OUTPUT_PATH = "/kaggle/working/" if IS_KAGGLE else "../output/"

if not IS_KAGGLE:
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
import os
import gc
import json
import torch
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy.sparse import lil_matrix
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

_CACHED_MODEL = None
_CACHED_TOKENIZER = None

def evaluate_principle(csv_filename, batch_size=1, max_length=2456, n_splits=5):
    global _CACHED_MODEL, _CACHED_TOKENIZER

    print(f"--- Starting Rigorous {n_splits}-Fold Evaluation for {csv_filename} ---")

    data_path = os.path.join(BASE_DATA_PATH, csv_filename)

    df = pd.read_csv(data_path)
    prompts = df['Prompt'].tolist()
    labels = np.array(df['GoldAnswer'].tolist())

    model_id = "meta-llama/Meta-Llama-3-8B"

    if _CACHED_MODEL is None or _CACHED_TOKENIZER is None:
        print("Loading model and tokenizer...")
        _CACHED_TOKENIZER = AutoTokenizer.from_pretrained(model_id, padding_side='left')
        if _CACHED_TOKENIZER.pad_token is None:
            _CACHED_TOKENIZER.pad_token = _CACHED_TOKENIZER.eos_token

        _CACHED_MODEL = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, device_map="auto"
        )
        _CACHED_MODEL.eval()

    tokenizer = _CACHED_TOKENIZER
    model = _CACHED_MODEL

    # =========================================================
    # 1. Hidden State Extraction (Last Token + float16)
    # =========================================================
    print("Extracting last-token hidden states (stored as float16 to save RAM)...")
    layer_features = {layer_idx: [] for layer_idx in range(0, 33)}

    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i : i + batch_size]

            inputs = tokenizer(
                batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_length
            ).to(model.device)

            outputs = model(**inputs, output_hidden_states=True)

            for layer_idx in range(0, 33):
                # Because padding_side='left', the last position [:, -1, :] is our true final token.
                # Cast to float16 to save massive amounts of Kaggle CPU RAM.
                last_token_hidden_state = outputs.hidden_states[layer_idx][:, -1, :].to(torch.float16).cpu().numpy()
                layer_features[layer_idx].append(last_token_hidden_state)

            del outputs, inputs
            torch.cuda.empty_cache()

    # =========================================================
    # 2. Hardened Classifier & CV Training
    # =========================================================
    print("Training regularized classifiers (C=0.1) with balanced weights...")
    avg_accuracies, std_accuracies = [], []
    avg_roc_aucs, std_roc_aucs = [], []
    layers = list(range(0, 33))

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for layer_idx in layers:
        # Cast to float32 right before sklearn training so the solvers don't crash
        X = np.vstack(layer_features[layer_idx]).astype(np.float32)

        fold_accs, fold_rocs = [], []

        for train_idx, test_idx in skf.split(X, labels):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = labels[train_idx], labels[test_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Upgraded Classifier for highly-dimensional representations
            clf = LogisticRegression(
                max_iter=2000,
                solver="liblinear",
                C=0.1,
                class_weight="balanced",
                random_state=42
            )
            clf.fit(X_train_scaled, y_train)

            preds = clf.predict(X_test_scaled)
            probs = clf.predict_proba(X_test_scaled)[:, 1]

            fold_accs.append(accuracy_score(y_test, preds))
            fold_rocs.append(roc_auc_score(y_test, probs))

        avg_accuracies.append(np.mean(fold_accs))
        std_accuracies.append(np.std(fold_accs))
        avg_roc_aucs.append(np.mean(fold_rocs))
        std_roc_aucs.append(np.std(fold_rocs))

    # =========================================================
    # 3. Output, DataFrame Export & Visualization
    # =========================================================
    best_acc_layer = int(np.argmax(avg_accuracies))
    print(f"\n--- Results ---")
    print(f"Best Layer for Accuracy: Layer {best_acc_layer} ({avg_accuracies[best_acc_layer]:.4f} ± {std_accuracies[best_acc_layer]:.4f})")

    results_df = pd.DataFrame({
        'Layer': layers,
        'Accuracy_Mean': avg_accuracies,
        'Accuracy_Std': std_accuracies,
        'ROC_AUC_Mean': avg_roc_aucs,
        'ROC_AUC_Std': std_roc_aucs
    })

    clean_name = csv_filename.replace('.csv', '')
    output_filename = os.path.join(OUTPUT_PATH, f"{clean_name}_cv_probing_results.csv")

    results_df.to_csv(output_filename, index=False)
    print(f"Saved robust results to: {output_filename}")

    # Plotting Mean with Error Bands (Standard Deviation)
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Avg Accuracy vs. Layer', 'Avg ROC-AUC vs. Layer'))

    fig.add_trace(go.Scatter(x=results_df['Layer'], y=results_df['Accuracy_Mean'], mode='lines+markers', name='Accuracy', marker=dict(color='blue'), error_y=dict(type='data', array=results_df['Accuracy_Std'], visible=True)), row=1, col=1)
    fig.add_trace(go.Scatter(x=results_df['Layer'], y=results_df['ROC_AUC_Mean'], mode='lines+markers', name='ROC-AUC', marker=dict(color='red'), error_y=dict(type='data', array=results_df['ROC_AUC_Std'], visible=True)), row=1, col=2)

    fig.update_layout(height=500, width=1000, title_text=f"Probing Llama-3-8B Representations: {csv_filename}", showlegend=False, hovermode="x unified")
    fig.show()
    fig.write_html(os.path.join(OUTPUT_PATH, f"{clean_name}_plot.html"))
    del layer_features, X, X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test
    gc.collect()
    torch.cuda.empty_cache()
    print("Ready for next file.\n")

    # Return the identified best layer index
    return best_acc_layer

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import gc
import json
import torch
import numpy as np
import pandas as pd
from scipy.sparse import lil_matrix
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

def extract_raw_sae_activations(csv_filename, best_layer, batch_size=1, max_length=2500):
    assert batch_size == 1, "Batch size must be 1 to prevent OOM errors."

    csv_path = f"/kaggle/input/datasets/shaurya01pratap/principledata/New Clean Data/{csv_filename}"
    df = pd.read_csv(csv_path)
    num_samples = len(df)

    print("Loading Llama 3 8B in 4-bit...")
    model_id = "meta-llama/Meta-Llama-3-8B"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=quantization_config,
        low_cpu_mem_usage=True
    )
    model.eval()

    print(f"Downloading SAE weights for layer {best_layer}...")
    sae_release = "EleutherAI/sae-llama-3-8b-32x"
    folder_name = f"layers.{best_layer}"

    cfg_path = hf_hub_download(repo_id=sae_release, filename=f"{folder_name}/cfg.json")
    weights_path = hf_hub_download(repo_id=sae_release, filename=f"{folder_name}/sae.safetensors")

    with open(cfg_path, "r") as f:
        raw_cfg = json.load(f)

    d_in = int(raw_cfg.get("d_in", 4096))
    expansion_factor = int(raw_cfg.get("expansion_factor", 32))
    k = int(raw_cfg.get("k", 192))
    d_sae = d_in * expansion_factor

    state_dict = load_file(weights_path)

    def get_weight(keys):
        for key in keys:
            if key in state_dict: return state_dict[key].to("cuda", dtype=torch.bfloat16)
        raise KeyError(f"Missing keys. Tried: {keys}")

    W_enc = get_weight(["W_enc", "encoder.weight"])
    b_enc = get_weight(["b_enc", "encoder.bias"])
    b_dec = get_weight(["b_dec", "decoder.bias"])

    if W_enc.shape[0] == d_sae:
        W_enc = W_enc.T

    print(f"Native PyTorch SAE initialized. d_sae={d_sae}, k={k}")

    # Sparse matrix to hold activations [num_samples, 131072] during the loop
    feature_acts_sparse = lil_matrix((num_samples, d_sae), dtype=np.float32)

    print(f"Starting extraction loop for {num_samples} prompts...")

    for i, row in df.iterrows():
        prompt = row['Prompt']

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            padding=True, # Dynamic padding (Fix 1 applied)
            truncation=True,
            max_length=max_length
        ).to(model.device) # Safe device mapping (Fix 2 applied)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # Correct layer indexing (Fix 3 maintained)
            last_token_state = outputs.hidden_states[best_layer + 1][:, -1, :].to(device=W_enc.device, dtype=W_enc.dtype)

            x_centered = last_token_state - b_dec
            pre_acts = torch.matmul(x_centered, W_enc) + b_enc
            vals, indices = torch.topk(pre_acts, k=k, dim=-1)

            # Move to CPU and format for NumPy
            vals = vals.squeeze(0).float().cpu().numpy()
            indices = indices.squeeze(0).cpu().numpy()

            valid_mask = vals > 0.0
            feature_acts_sparse[i, indices[valid_mask]] = vals[valid_mask]

        del inputs, outputs, last_token_state, x_centered, pre_acts, vals, indices
        torch.cuda.empty_cache()
        gc.collect()

        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{num_samples} prompts.")

    print("Transposing matrix to [131072 rows x 1793 columns]...")
    # Transpose so rows are features, columns are prompts
    acts_transposed = feature_acts_sparse.T.tocsc()

    print("Converting to dense DataFrame (this requires ~1GB of RAM)...")
    col_names = [f"Prompt_{i}" for i in range(num_samples)]
    df_acts = pd.DataFrame(acts_transposed.toarray(), columns=col_names)

    # Insert Feature_ID as the very first column
    df_acts.insert(0, "Feature_ID", np.arange(d_sae))
    output_filename = f"/kaggle/working/Raw_Activations_{csv_filename.replace('.csv', '')}_Layer_{best_layer}.csv"
    #output_filename = f"/content/drive/MyDrive/Raw_Activations_{csv_filename.replace('.csv', '')}_Layer_{best_layer}.csv"
    print(f"Saving to {output_filename}...")
    df_acts.to_csv(output_filename, index=False)

    print("Done! Pipeline complete.")

    del model, W_enc, b_enc, b_dec, feature_acts_sparse, acts_transposed, df_acts
    torch.cuda.empty_cache()
    gc.collect()

    return output_filename

In [ ]:
def save_raw_activations(principle_file_name):
  best_layer = evaluate_principle(csv_filename=principle_file_name, batch_size=1, max_length=2456, n_splits=5)
  extract_raw_sae_activations(csv_filename=principle_file_name, best_layer=best_layer, batch_size=1, max_length=2500)

In [ ]:
for i in range(4,5):
  my_csv = f"Principle{i}.csv"
  save_raw_activations(my_csv)